<a href="https://colab.research.google.com/github/sangramshitole07/PNLP/blob/main/PNL_Lab04_Sangram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Principles of Natural Language Processing Lab  
## Language Modelling: N-grams, N-gram Probabilities, Evaluation and Perplexity

**Course:** AME 5053 — Principles of Natural Language Processing Lab  
**Lab duration:** 3 hours  
**Mode:** Guided implementation + comparison + interpretation

### Learning outcomes
By the end of this lab, you should be able to:

1. construct unigram, bigram and trigram language models from a text corpus;
2. estimate N-gram probabilities using maximum-likelihood estimation (MLE);
3. compute the probability of a sentence under an N-gram model;
4. explain the effect of sentence-boundary symbols such as `<s>` and `</s>`;
5. evaluate a language model on unseen text using perplexity;
6. diagnose why an unsmoothed N-gram model can fail on unseen N-grams.

### What you must submit
Your notebook should contain:
- your predictions before running selected cells;
- completed implementations;
- at least one comparison between models;
- an error analysis using concrete examples;
- a short conclusion explaining what model behaviour you observed.

## 1. Corpus used in this lab

We will use a small, controlled corpus so that the counts can be inspected manually.

```text
students learn natural language processing
students learn machine learning
students study language models
language models predict words
language models assign probabilities
machine learning models learn patterns
natural language processing uses language models
```

The corpus is intentionally small. This makes the probability calculations transparent, but it will also expose an important limitation: **many valid word sequences will never occur in the training corpus**.

We will later split examples into **training** and **test** sentences so that perplexity is computed on data that the model did not simply memorize.

In [1]:
from collections import Counter, defaultdict
import math
import pandas as pd

raw_sentences = [
    "students learn natural language processing",
    "students learn machine learning",
    "students study language models",
    "language models predict words",
    "language models assign probabilities",
    "machine learning models learn patterns",
    "natural language processing uses language models",
]

raw_sentences

['students learn natural language processing',
 'students learn machine learning',
 'students study language models',
 'language models predict words',
 'language models assign probabilities',
 'machine learning models learn patterns',
 'natural language processing uses language models']

## 1.1 Tokenize the corpus
Run the helper below and inspect the output.

In [2]:
def tokenize(sentence):
    return sentence.lower().split()

tokenized = [tokenize(s) for s in raw_sentences]
tokenized

[['students', 'learn', 'natural', 'language', 'processing'],
 ['students', 'learn', 'machine', 'learning'],
 ['students', 'study', 'language', 'models'],
 ['language', 'models', 'predict', 'words'],
 ['language', 'models', 'assign', 'probabilities'],
 ['machine', 'learning', 'models', 'learn', 'patterns'],
 ['natural', 'language', 'processing', 'uses', 'language', 'models']]

## 2. Sentence boundaries

A language model should know not only which words can follow other words, but also where a sentence can begin and end.

For a bigram model we will represent:

```text
students learn machine learning
```

as

```text
<s> students learn machine learning </s>
```

For a trigram model, we need enough history at the beginning. We therefore use two start symbols:

```text
<s> <s> students learn machine learning </s>
```

This convention allows the model to estimate probabilities for sentence beginnings as well as sentence endings.

### Prediction 1 — before coding

Without running any code, answer:

1. Which word do you expect to be the most frequent unigram?
2. Which bigram do you expect to have a relatively high probability?
3. Do you expect a trigram model to assign non-zero probability to more or fewer unseen test sentences than a bigram model? Why?

Write your answers below.

**Your prediction:**  
- Most frequent unigram:  
- Likely high-probability bigram:  
- Bigram vs trigram on unseen text:

## 3. Build N-gram counts

For an N-gram model, we count neighbouring sequences of words.

For example, in:

```text
language models predict words
```

the bigrams are:

```text
(<s>, language)
(language, models)
(models, predict)
(predict, words)
(words, </s>)
```

The trigram model uses sequences of three tokens.

We will use these counts to estimate conditional probabilities.

In [3]:
def add_boundaries(tokens, n):
    # TODO:
    # Bigram model: add one <s> and one </s>
    if n==1:
      return list(tokens)
    return ["<s>"] * (n-1) + list(tokens)+['</s>']
    # Trigram model: add two <s> symbols and one </s>

print(add_boundaries(['students','learn'],2))
print(add_boundaries(['students','learn'],3))



['<s>', 'students', 'learn', '</s>']
['<s>', '<s>', 'students', 'learn', '</s>']


In [4]:
def make_ngrams(tokens, n):
    # TODO: return all adjacent n-grams as tuples
    # Example: ['a','b','c'], n=2 -> [('a','b'), ('b','c')]
    return[tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
print(make_ngrams(['a','b','c'],2))

[('a', 'b'), ('b', 'c')]


In [5]:
# TODO: build unigram, bigram and trigram counts over the whole corpus.
# Suggested names:
# unigram_counts
# bigram_counts
# trigram_counts

unigram_counts = Counter()
bigram_counts = Counter()
trigram_counts = Counter()
for sentence in tokenized:
  unigram_counts.update(sentence)
  bigram_counts.update(make_ngrams(add_boundaries(sentence,2),2))
  trigram_counts.update(make_ngrams(add_boundaries(sentence,3),3))
print('Unigram counts',len(unigram_counts))
print('bigram counts',len(bigram_counts))
print('Trigram counts',len(trigram_counts))

Unigram counts 15
bigram counts 28
Trigram counts 32


In [6]:
def top_table(counter, k=10):
  return pd.DataFrame(counter.most_common(k), columns=['N-gram', 'count'])
display(top_table(unigram_counts))
display(top_table(bigram_counts))
display(top_table(trigram_counts))

,N-gram,count
0,language,6
1,models,5
2,students,3
3,learn,3
4,natural,2
5,processing,2
6,machine,2
7,learning,2
8,study,1
9,predict,1


,N-gram,count
0,"(language, models)",4
1,"(<s>, students)",3
2,"(students, learn)",2
3,"(natural, language)",2
4,"(language, processing)",2
5,"(machine, learning)",2
6,"(models, </s>)",2
7,"(<s>, language)",2
8,"(learn, natural)",1
9,"(processing, </s>)",1


,N-gram,count
0,"(<s>, <s>, students)",3
1,"(<s>, students, learn)",2
2,"(natural, language, processing)",2
3,"(language, models, </s>)",2
4,"(<s>, <s>, language)",2
5,"(<s>, language, models)",2
6,"(students, learn, natural)",1
7,"(learn, natural, language)",1
8,"(language, processing, </s>)",1
9,"(students, learn, machine)",1


In [7]:
bigram_counts

Counter({('<s>', 'students'): 3,
         ('students', 'learn'): 2,
         ('learn', 'natural'): 1,
         ('natural', 'language'): 2,
         ('language', 'processing'): 2,
         ('processing', '</s>'): 1,
         ('learn', 'machine'): 1,
         ('machine', 'learning'): 2,
         ('learning', '</s>'): 1,
         ('students', 'study'): 1,
         ('study', 'language'): 1,
         ('language', 'models'): 4,
         ('models', '</s>'): 2,
         ('<s>', 'language'): 2,
         ('models', 'predict'): 1,
         ('predict', 'words'): 1,
         ('words', '</s>'): 1,
         ('models', 'assign'): 1,
         ('assign', 'probabilities'): 1,
         ('probabilities', '</s>'): 1,
         ('<s>', 'machine'): 1,
         ('learning', 'models'): 1,
         ('models', 'learn'): 1,
         ('learn', 'patterns'): 1,
         ('patterns', '</s>'): 1,
         ('<s>', 'natural'): 1,
         ('processing', 'uses'): 1,
         ('uses', 'language'): 1})

### Inspect the most frequent N-grams
Create tables showing the 10 most frequent unigrams, bigrams and trigrams.

In [8]:
# TODO: display the 10 most frequent items from each Counter.
print("10 most frequent items in unigram")
unigram_counts.most_common(10)

10 most frequent items in unigram


[('language', 6),
 ('models', 5),
 ('students', 3),
 ('learn', 3),
 ('natural', 2),
 ('processing', 2),
 ('machine', 2),
 ('learning', 2),
 ('study', 1),
 ('predict', 1)]

In [9]:
print("10 most frequent items in bigram")
bigram_counts.most_common(10)

10 most frequent items in bigram


[(('language', 'models'), 4),
 (('<s>', 'students'), 3),
 (('students', 'learn'), 2),
 (('natural', 'language'), 2),
 (('language', 'processing'), 2),
 (('machine', 'learning'), 2),
 (('models', '</s>'), 2),
 (('<s>', 'language'), 2),
 (('learn', 'natural'), 1),
 (('processing', '</s>'), 1)]

In [10]:
print("10 most frequent items in trigram")
trigram_counts.most_common(10)

10 most frequent items in trigram


[(('<s>', '<s>', 'students'), 3),
 (('<s>', 'students', 'learn'), 2),
 (('natural', 'language', 'processing'), 2),
 (('language', 'models', '</s>'), 2),
 (('<s>', '<s>', 'language'), 2),
 (('<s>', 'language', 'models'), 2),
 (('students', 'learn', 'natural'), 1),
 (('learn', 'natural', 'language'), 1),
 (('language', 'processing', '</s>'), 1),
 (('students', 'learn', 'machine'), 1)]

In [11]:
bigram_counts.items()

dict_items([(('<s>', 'students'), 3), (('students', 'learn'), 2), (('learn', 'natural'), 1), (('natural', 'language'), 2), (('language', 'processing'), 2), (('processing', '</s>'), 1), (('learn', 'machine'), 1), (('machine', 'learning'), 2), (('learning', '</s>'), 1), (('students', 'study'), 1), (('study', 'language'), 1), (('language', 'models'), 4), (('models', '</s>'), 2), (('<s>', 'language'), 2), (('models', 'predict'), 1), (('predict', 'words'), 1), (('words', '</s>'), 1), (('models', 'assign'), 1), (('assign', 'probabilities'), 1), (('probabilities', '</s>'), 1), (('<s>', 'machine'), 1), (('learning', 'models'), 1), (('models', 'learn'), 1), (('learn', 'patterns'), 1), (('patterns', '</s>'), 1), (('<s>', 'natural'), 1), (('processing', 'uses'), 1), (('uses', 'language'), 1)])

## 4. Maximum-likelihood estimation of N-gram probabilities

For a bigram model,

$$
P(w_i \mid w_{i-1})
=
\frac{C(w_{i-1}, w_i)}{C(w_{i-1})}
$$

For a trigram model,

$$
P(w_i \mid w_{i-2}, w_{i-1})
=
\frac{C(w_{i-2}, w_{i-1}, w_i)}
     {C(w_{i-2}, w_{i-1})}
$$

The key idea is simple: **among all occurrences of the history, how often was the next word the one we are asking about?**

This is a maximum-likelihood estimate because the probabilities are obtained directly from observed frequencies in the training corpus.

In [12]:
bigram_context_counts = Counter()
for (w1,w2), c in bigram_counts.items():
    bigram_context_counts[w1] += c
trigram_context_counts = Counter()
for (w1,w2,w3), c in trigram_counts.items():
    trigram_context_counts[(w1,w2)] += c
def bigram_probability(w1, w2, bigram_counts, unigram_context_counts):
    # TODO: implement P(w2 | w1)
    den=unigram_context_counts[w1]
    if den==0:
      return 0.0
    res = bigram_counts[(w1,w2)]/den
    return res

def trigram_probability(w1, w2, w3, trigram_counts, bigram_context_counts):
    # TODO: implement P(w3 | w1,w2)
    den=bigram_context_counts[(w1,w2)]
    if den==0:
      return 0.0
    res = trigram_counts[(w1,w2,w3)]/den
    return res

### Probability checks
Compute and interpret the following:

- `P(learn | students)`
- `P(models | language)`
- `P(language | learn)`
- `P(models | study, language)`

In [13]:
# TODO: create the context counts needed by the probability functions,
checks={
    'p(learn|students)':bigram_probability('students','learn',bigram_counts,bigram_context_counts),
    'p(models|language)':bigram_probability('language','models',bigram_counts,bigram_context_counts),
    'p(language|learn)':bigram_probability('learn','language',bigram_counts,bigram_context_counts),
    'p(models|study,language)':trigram_probability('study','language','models',trigram_counts,trigram_context_counts)
}
# then compute the four probabilities above.

## 5. Sentence probability

Under a bigram model, the probability of a sentence is approximated using the Markov assumption:

$$
P(w_1,\ldots,w_m)
\approx
P(w_1\mid <s>)
\prod_{i=2}^{m}P(w_i\mid w_{i-1})
P(</s>\mid w_m)
$$
The trigram model conditions each word on the previous two tokens.

Because sentence probabilities are products of many values smaller than 1, they can become extremely small. In larger systems we therefore often work with **log probabilities**.

In [22]:
def bigram_sentence_probability(sentence, bigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add boundaries
    # 3. multiply bigram probabilities
    toks=add_boundaries(tokenize(sentence),2)
    prob =1.0
    for w1,w2 in make_ngrams(toks,2):
      prob*=bigram_probability(w1,w2,bigram_counts,context_counts)
    return prob

def trigram_sentence_probability(sentence, trigram_counts, context_counts):
    # TODO:
    # 1. tokenize
    # 2. add trigram boundaries
    # 3. multiply trigram probabilities
    toks=add_boundaries(tokenize(sentence),3)
    prob =1.0
    for w1,w2,w3 in make_ngrams(toks,3):
      prob*=trigram_probability(w1,w2,w3,trigram_counts,context_counts)
    return prob


### Compare sentence probabilities
Evaluate these two training-like sentences:

- `students learn machine learning`
- `language models assign probabilities`

Then explain why their probabilities differ.

In [23]:
for s in[
    "Students learn machine learning",
    "Language models assign probabilities",
    "Students predict probabilities"
]:

  print("\n", s)
  print("Bigram probability:", bigram_sentence_probability(s, bigram_counts, bigram_context_counts))
  print("Trigram probability:", trigram_sentence_probability(s, trigram_counts, trigram_context_counts))


 Students learn machine learning
Bigram probability: 0.047619047619047616
Trigram probability: 0.07142857142857142

 Language models assign probabilities
Bigram probability: 0.0380952380952381
Trigram probability: 0.07142857142857142

 Students predict probabilities
Bigram probability: 0.0
Trigram probability: 0.0


## 6. Evaluation on unseen text

A useful language model should perform well on text that was **not used to estimate its probabilities**.

We will use:

### Test sentence A
`students learn language models`

### Test sentence B
`language models learn patterns`

### Test sentence C
`students predict probabilities`

Before computing anything, inspect the corpus and predict which test sentences may contain unseen bigrams or trigrams.

### Prediction 2 — unseen sequences

Before running the next section, list at least one bigram/trigram that you think is unseen in each test sentence.

**A. students learn language models**  
Prediction:

**B. language models learn patterns**  
Prediction:

**C. students predict probabilities**  
Prediction:

## 7. Perplexity

Perplexity is a standard intrinsic evaluation measure for language models.

For a sequence containing \(N\) predicted tokens,

$$
PP(W)
=
P(W)^{-1/N}
$$

Equivalently, using log probabilities,

$$
PP(W)
=
\exp\left(
-\frac{1}{N}
\sum_{i=1}^{N}\log P(w_i \mid \text{history})
\right)
$$

Interpretation:

- **lower perplexity** means the model assigns higher probability to the observed test sequence;
- **higher perplexity** means the sequence is more surprising to the model;
- if an unsmoothed model encounters an N-gram with probability zero, the perplexity becomes infinite.

Perplexity should only be compared meaningfully when models are evaluated on the **same tokenization and same test data**.

In [24]:
def bigram_perplexity(sentence, bigram_counts, context_counts):
    # Use log probabilities.
    # If any required bigram has probability 0, return math.inf.
    toks = tokenize(sentence)
    toks_with_boundaries = add_boundaries(toks, 2)
    ngrams = make_ngrams(toks_with_boundaries, 2)

    total_log_prob = 0.0
    N = len(toks) + 1 # Number of predicted tokens (words + </s>)

    for i in range(len(ngrams)):
        w1, w2 = ngrams[i]
        prob = bigram_probability(w1, w2, bigram_counts, context_counts)
        if prob == 0.0:
            return math.inf
        total_log_prob += math.log(prob)

    return math.exp(-total_log_prob / N)

def trigram_perplexity(sentence, trigram_counts, context_counts):
    # Use log probabilities.
    # If any required trigram has probability 0, return math.inf.
    toks = tokenize(sentence)
    toks_with_boundaries = add_boundaries(toks, 3)
    ngrams = make_ngrams(toks_with_boundaries, 3)

    total_log_prob = 0.0
    N = len(toks) + 1 # Number of predicted tokens (words + </s>)

    for i in range(len(ngrams)):
        w1, w2, w3 = ngrams[i]
        prob = trigram_probability(w1, w2, w3, trigram_counts, context_counts)
        if prob == 0.0:
            return math.inf
        total_log_prob += math.log(prob)

    return math.exp(-total_log_prob / N)

In [25]:
test_sentences = [
    "students learn language models",
    "language models learn patterns",
    "students predict probabilities",
]

perplexities = []
for sentence in test_sentences:
    bg_perp = bigram_perplexity(sentence, bigram_counts, bigram_context_counts)
    tg_perp = trigram_perplexity(sentence, trigram_counts, trigram_context_counts)
    perplexities.append({
        "sentence": sentence,
        "bigram_perplexity": bg_perp,
        "trigram_perplexity": tg_perp
    })

perplexity_df = pd.DataFrame(perplexities)
display(perplexity_df)

,sentence,bigram_perplexity,trigram_perplexity
0,students learn language models,inf,inf
1,language models learn patterns,2.394694,inf
2,students predict probabilities,inf,inf


## 8. Diagnose the model
For every test sentence with infinite perplexity, print the first unseen N-gram that causes the failure.

In [21]:
def first_unseen_ngram(sentence, n, ngram_counts):
    toks = tokenize(sentence)
    toks_with_boundaries = add_boundaries(toks, n)
    ngrams = make_ngrams(toks_with_boundaries, n)

    for ngram in ngrams:
        if ngram_counts[ngram] == 0:
            return ngram
    return None

print("Diagnosing infinite perplexities:")
for index, row in perplexity_df.iterrows():
    sentence = row['sentence']
    if row['bigram_perplexity'] == math.inf:
        unseen_bigram = first_unseen_ngram(sentence, 2, bigram_counts)
        print(f"Sentence: '{sentence}' - First unseen bigram: {unseen_bigram}")
    if row['trigram_perplexity'] == math.inf:
        unseen_trigram = first_unseen_ngram(sentence, 3, trigram_counts)
        print(f"Sentence: '{sentence}' - First unseen trigram: {unseen_trigram}")

Diagnosing infinite perplexities:
Sentence: 'students learn language models' - First unseen bigram: ('learn', 'language')
Sentence: 'students learn language models' - First unseen trigram: ('students', 'learn', 'language')
Sentence: 'language models learn patterns' - First unseen trigram: ('language', 'models', 'learn')
Sentence: 'students predict probabilities' - First unseen bigram: ('students', 'predict')
Sentence: 'students predict probabilities' - First unseen trigram: ('<s>', 'students', 'predict')


## 9. Interpretation and error analysis

Answer in complete sentences.

1. Which model gave lower perplexity on the test sentences that both models could score?
2. Did the higher-order model always perform better? Explain using the size of this corpus.
3. Identify one unseen bigram and one unseen trigram from the test set.
4. Why does a zero probability create a serious problem when sentence probability is computed by multiplication?
5. What would you expect to happen if the training corpus became much larger?

1. **Which model gave lower perplexity on the test sentences that both models could score?**
    Only 'language models learn patterns' could be scored by both models (bigram perplexity: 2.394694, trigram perplexity: inf). So, only bigram model successfully scored this sentence. Thus, the bigram model gave a lower perplexity here.
2. **Did the higher-order model always perform better? Explain using the size of this corpus.**
    The higher-order model (trigram) did not always perform better. In fact, for two out of three sentences, the trigram model had infinite perplexity, whereas the bigram model had infinite perplexity for only two sentences. This is likely due to the small size of the corpus. A higher-order model like a trigram model requires more data to see specific sequences of three words. If a trigram combination hasn't been observed in the training data, its probability will be zero, leading to infinite perplexity.
3. **Identify one unseen bigram and one unseen trigram from the test set.**
    *   One unseen bigram: `('learn', 'language')` from the sentence "students learn language models".
    *   One unseen trigram: `('students', 'learn', 'language')` from the sentence "students learn language models".
4. **Why does a zero probability create a serious problem when sentence probability is computed by multiplication?**
    When computing sentence probability by multiplying individual N-gram probabilities, if even one N-gram probability is zero, the entire sentence probability becomes zero. This leads to an undefined logarithm in the perplexity calculation (log(0)), which then results in infinite perplexity. This means the model assigns no likelihood to the sentence, deeming it impossible.
5. **What would you expect to happen if the training corpus became much larger?**
    If the training corpus became much larger, I would expect the perplexity to decrease for both bigram and trigram models. A larger corpus would mean a higher chance of observing more N-grams, reducing the occurrence of unseen N-grams and thus fewer instances of zero probabilities. This would generally lead to more robust probability estimates and lower, finite perplexity values, especially for the trigram model, which is more sensitive to data sparsity.

## 10. Viva / discussion questions

1. Why is an N-gram language model called a probabilistic model?
2. What is the Markov assumption in a bigram model?
3. Why are sentence boundary symbols useful?
4. Why can a trigram model be more data-hungry than a bigram model?
5. Why is perplexity usually preferred over raw sentence probability when comparing sequences of different lengths?
6. Why can an unsmoothed N-gram model assign infinite perplexity to a perfectly grammatical sentence?